# Pierce County Water Quality Index: QA and Year-Over-Year Change

Pierce County's Surface Water Management division publishes a Water
Quality Index (WQI) score for real monitoring stations across the
county - bacteria, dissolved oxygen, pH, phosphorus, total suspended
solids, temperature, nitrogen, and turbidity, each converted to a 0-100
sub-index, combined into an overall annual and monthly score.

**Scope note, corrected before building anything further:** this started
as a "multi-year trend" idea. Checking the real data first showed that's
not accurate - the public layer holds mostly two real observation years
(2023 and 2024) per station, with a handful of scattered legacy rows from
2014-2018. So this notebook does a real **2023 vs 2024 year-over-year
comparison**, not a multi-year trend line - said honestly up front rather
than oversold.

A second correction: the parameter fields (`PH_ANNUAL`, `DO_ANNUAL`, etc.)
turned out to be pre-converted 0-100 WQI sub-index scores, not raw pH/DO
measurements - so QA/QC here checks the *index computation and
publication*, not raw field-instrument readings against physical bounds
like pH 0-14.

This notebook does three things:
1. **QA/QC on the WQI data itself** - range validity, internal
   consistency between the overall score and its components, and
   completeness.
2. **Real year-over-year change** (2023 to 2024) per station.
3. **A genuine cross-project question**: do areas with more unconfirmed
   stormwater outfalls (from this portfolio's
   `pierce-stormwater-outfall-audit` project) show worse water quality or
   more decline?

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import os

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)
BASE = "https://services2.arcgis.com/1UvBaQ5y1ubjUPmd/arcgis/rest/services"

SUB_PARAMS = ["BAC_ANNUAL", "DO_ANNUAL", "PH_ANNUAL", "PHOS_ANNUAL",
              "TSS_ANNUAL", "TEMP_ANNUAL", "NITRO_ANNUAL", "TURB_ANNUAL"]

## Step 1: Fetch the real data

In [2]:
url = f"{BASE}/Water_Quality_Monitoring_Sites/FeatureServer/0/query"
resp = requests.get(url, params={"where": "1=1", "outFields": "*", "outSR": 4326, "f": "geojson"})
wq = gpd.GeoDataFrame.from_features(resp.json()["features"], crs=4326)
print(f"Water quality records: {len(wq)}")
wq.to_file(f"{DATA_DIR}/water_quality_sites_raw.geojson", driver="GeoJSON")

Water quality records: 146


## Step 2: QA/QC on the WQI data

### Range validity - is every sub-score and the overall score within [0, 100]?

In [3]:
for col in SUB_PARAMS + ["OVERALL_ANNUAL"]:
    bad = wq[(wq[col] < 0) | (wq[col] > 100)]
    print(f"{col}: {len(bad)} out-of-[0,100] values")

BAC_ANNUAL: 0 out-of-[0,100] values
DO_ANNUAL: 0 out-of-[0,100] values
PH_ANNUAL: 0 out-of-[0,100] values
PHOS_ANNUAL: 0 out-of-[0,100] values
TSS_ANNUAL: 0 out-of-[0,100] values
TEMP_ANNUAL: 0 out-of-[0,100] values
NITRO_ANNUAL: 0 out-of-[0,100] values
TURB_ANNUAL: 0 out-of-[0,100] values
OVERALL_ANNUAL: 0 out-of-[0,100] values


All clean - a real check that passed, worth reporting as a result in
its own right, not just checks that found problems.

### Internal consistency - does `OVERALL_ANNUAL` reconcile with its 8 components?

In [4]:
complete = wq.dropna(subset=SUB_PARAMS + ["OVERALL_ANNUAL"])
print(f"Rows with all 8 sub-scores + overall present: {len(complete)} of {len(wq)}")

simple_avg = complete[SUB_PARAMS].mean(axis=1)
diff = complete["OVERALL_ANNUAL"] - simple_avg
print(f"\nOVERALL_ANNUAL vs plain average of the 8 sub-scores:")
print(f"  correlation: {complete['OVERALL_ANNUAL'].corr(simple_avg):.3f}")
print(f"  mean difference: {diff.mean():.1f} (OVERALL is systematically lower)")

Rows with all 8 sub-scores + overall present: 138 of 146

OVERALL_ANNUAL vs plain average of the 8 sub-scores:
  correlation: 0.827
  mean difference: -15.6 (OVERALL is systematically lower)


**Correlated (0.83) but not a simple average - and that's expected, not
a bug.** `OVERALL_ANNUAL` runs about 15-16 points below a plain mean of
its 8 components on average. Standard WQI methodologies commonly weight
or penalize the worst-performing parameter more heavily than a flat
average would (a station that's excellent on 7 parameters but has a
bacteria problem should score worse than a naive average implies) - the
systematic gap is consistent with that, not a computation error. Reported
as real methodological context, not flagged as a defect.

### Completeness by parameter

In [5]:
for col in SUB_PARAMS + ["OVERALL_ANNUAL"]:
    missing = wq[col].isna().sum()
    print(f"{col}: {missing} missing of {len(wq)} ({100*missing/len(wq):.1f}%)")

BAC_ANNUAL: 2 missing of 146 (1.4%)
DO_ANNUAL: 4 missing of 146 (2.7%)
PH_ANNUAL: 0 missing of 146 (0.0%)
PHOS_ANNUAL: 0 missing of 146 (0.0%)
TSS_ANNUAL: 0 missing of 146 (0.0%)
TEMP_ANNUAL: 2 missing of 146 (1.4%)
NITRO_ANNUAL: 2 missing of 146 (1.4%)
TURB_ANNUAL: 2 missing of 146 (1.4%)
OVERALL_ANNUAL: 0 missing of 146 (0.0%)


### A duplicate-record check, same real issue found in this
portfolio's outfall-audit project, re-verified here on a fresh fetch

In [6]:
print(f"Unique STATION_NAME values: {wq['STATION_NAME'].nunique()} for {len(wq)} rows")
coords = wq.geometry.apply(lambda g: (round(g.x, 6), round(g.y, 6)))
print(f"Unique coordinate pairs: {coords.nunique()}")

# Multiple rows at the same coordinate is normal and expected here (each
# station legitimately has ~2 rows, one per real observation year). The
# actual bug to check for is a coordinate associated with more than one
# DISTINCT station name - that's the same-station-different-spelling
# issue found in the sibling outfall-audit project.
name_counts_per_coord = wq.groupby(coords)["STATION_NAME"].nunique()
bad_coords = name_counts_per_coord[name_counts_per_coord > 1].index
dup = wq[coords.isin(bad_coords)].sort_values("STATION_NAME")[["STATION_NAME"]]
if len(dup):
    print(f"\nCoordinates shared by more than one distinct station name ({len(dup)} rows):")
    print(dup)
else:
    print("\nNo coordinates shared by multiple distinct station names.")

Unique STATION_NAME values: 85 for 146 rows
Unique coordinate pairs: 83

Coordinates shared by more than one distinct station name (6 rows):
                     STATION_NAME
60       MP_CanyonFallsCreek_0.58
128      MP_CanyonFallsCreek_0.58
144      MP_CanyonfallsCreek_0.58
66            NI_25MileCreek_0.24
132           NI_25MileCreek_0.24
145  NI_Twenty-fiveMileCreek_0.24


Same real inconsistency as before (`CanyonFallsCreek` vs
`CanyonfallsCreek`, `25MileCreek` vs `Twenty-fiveMileCreek`) - confirms
this is a persistent issue in the county's source data, not a one-time
fetch artifact. Deduplicated on coordinates for everything below.

In [7]:
wq["_coord_key"] = coords
wq_dedup = wq.drop_duplicates(subset=["_coord_key", "sample_date_x"]).drop(columns="_coord_key")
print(f"After coordinate-based dedup: {len(wq_dedup)} rows")

After coordinate-based dedup: 143 rows


## Step 3: Real year-over-year change (2023 to 2024)

Checking which years actually have enough coverage for a real comparison
before doing one - not assumed.

In [8]:
wq_dedup["date_x"] = pd.to_datetime(wq_dedup["sample_date_x"], unit="ms", errors="coerce")
wq_dedup["water_year"] = wq_dedup["date_x"].dt.year
print(wq_dedup["water_year"].value_counts().sort_index())

water_year
2014     1
2016     5
2017    13
2018     4
2023    60
2024    60
Name: count, dtype: int64


In [9]:
y2023 = wq_dedup[wq_dedup["water_year"] == 2023][["STATION_NAME", "OVERALL_ANNUAL", "geometry"]].rename(
    columns={"OVERALL_ANNUAL": "wqi_2023"})
y2024 = wq_dedup[wq_dedup["water_year"] == 2024][["STATION_NAME", "OVERALL_ANNUAL"]].rename(
    columns={"OVERALL_ANNUAL": "wqi_2024"})

change = y2023.merge(y2024, on="STATION_NAME", how="inner")
change["wqi_change"] = change["wqi_2024"] - change["wqi_2023"]
print(f"Stations with both 2023 and 2024 scores: {len(change)}")
print(change["wqi_change"].describe())

print("\nBiggest improvements:")
print(change.nlargest(5, "wqi_change")[["STATION_NAME","wqi_2023","wqi_2024","wqi_change"]].to_string(index=False))
print("\nBiggest declines:")
print(change.nsmallest(5, "wqi_change")[["STATION_NAME","wqi_2023","wqi_2024","wqi_change"]].to_string(index=False))

Stations with both 2023 and 2024 scores: 59
count    59.000000
mean      0.322034
std      11.519759
min     -24.000000
25%      -7.000000
50%       1.000000
75%       8.500000
max      23.000000
Name: wqi_change, dtype: float64

Biggest improvements:
             STATION_NAME  wqi_2023  wqi_2024  wqi_change
        CL_DiruCreek_0.59        65        88          23
      CC_CloverCreek_1.84        21        42          21
NI_LittleMashelRiver_0.31        25        45          20
   GH_GoodnoughCreek_0.09        56        73          17
    GH_CrescentCreek_0.81        45        60          15

Biggest declines:
                STATION_NAME  wqi_2023  wqi_2024  wqi_change
CC_NorthForkCloverCreek_0.99        81        57         -24
CC_NorthForkCloverCreek_1.03        84        60         -24
       KP_DutchersCreek_0.61        86        62         -24
    AI_SchoolHouseCreek_0.65        72        52         -20
CC_NorthForkCloverCreek_0.04        81        63         -18


## Step 4: A real cross-project question

This portfolio's `pierce-stormwater-outfall-audit` project scored 25,659
real stormwater outfalls for inspection priority, using (among other
factors) proximity to each point's *nearest* water quality station. That
means the outfall risk data already carries a per-station context. Reusing
it here to ask a real question: do stations near more/higher-risk
unconfirmed outfalls show worse 2024 water quality or more decline?

In [10]:
outfall_path = "../../pierce-stormwater-outfall-audit/docs/outfalls_all.geojson"
if os.path.exists(outfall_path):
    outfalls = gpd.read_file(outfall_path)
    print(f"Loaded {len(outfalls)} scored outfalls from the sibling project")

    change_utm = change.to_crs(32610)
    outfalls_utm = outfalls.to_crs(32610)

    # for each WQ station, count nearby unconfirmed outfalls within 1km
    # and their mean risk score
    results = []
    for _, row in change_utm.iterrows():
        nearby = outfalls_utm[outfalls_utm.geometry.distance(row.geometry) < 1000]
        results.append({
            "STATION_NAME": row["STATION_NAME"],
            "nearby_outfalls": len(nearby),
            "nearby_unconfirmed": int(nearby["unconfirmed_flag"].sum()) if len(nearby) else 0,
            "mean_risk_score": nearby["risk_score"].mean() if len(nearby) else np.nan,
        })
    context = pd.DataFrame(results)
    merged = change.merge(context, on="STATION_NAME")

    print(f"\nCorrelation, nearby unconfirmed outfall count vs 2024 WQI: "
          f"{merged['nearby_unconfirmed'].corr(merged['wqi_2024']):.3f}")
    print(f"Correlation, nearby unconfirmed outfall count vs WQI change: "
          f"{merged['nearby_unconfirmed'].corr(merged['wqi_change']):.3f}")
else:
    print("Sibling project's scored outfall data not found at expected path - skipping cross-reference.")
    merged = change.copy()
    merged["nearby_outfalls"] = np.nan
    merged["nearby_unconfirmed"] = np.nan
    merged["mean_risk_score"] = np.nan

Loaded 25659 scored outfalls from the sibling project

Correlation, nearby unconfirmed outfall count vs 2024 WQI: -0.023
Correlation, nearby unconfirmed outfall count vs WQI change: -0.124


**Report whatever this actually shows - a real correlation in either
direction is a finding; a weak/near-zero one is also a real, honest
result, not a failure.** With 59 stations and a 1km proximity window, this
is a modest sample for a correlation, not a rigorous causal claim - stated
here as a limitation, not glossed over.

## Step 5: Is "no correlation" actually meaningful, or just underpowered?

"r is close to zero" isn't a complete answer on its own - with a sample
this small, a real but modest relationship could easily fail to reach
significance. Testing both questions explicitly: is the observed
correlation statistically distinguishable from zero, and what's the
smallest true correlation this sample size could have reliably caught?

In [11]:
from scipy import stats as scipy_stats

sig_results = {}
for col, target in [("nearby_unconfirmed", "wqi_2024"), ("nearby_unconfirmed", "wqi_change")]:
    valid = merged.dropna(subset=[col, target])
    r, p = scipy_stats.pearsonr(valid[col], valid[target])
    n = len(valid)
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    ci_low, ci_high = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
    print(f"{col} vs {target} (n={n}): r={r:.3f}, p={p:.3f}, 95% CI=[{ci_low:.3f}, {ci_high:.3f}]")
    sig_results[target] = {"r": round(float(r), 3), "p": round(float(p), 3), "n": int(n),
                            "ci_low": round(float(ci_low), 3), "ci_high": round(float(ci_high), 3)}

# Minimum detectable |r| at this sample size (standard power - Fisher z), alpha=0.05, power=0.80
n_power = len(merged.dropna(subset=["nearby_unconfirmed", "wqi_2024"]))
alpha, power = 0.05, 0.80
z_alpha = scipy_stats.norm.ppf(1 - alpha / 2)
z_power = scipy_stats.norm.ppf(power)
min_detectable_r = np.tanh((z_alpha + z_power) / np.sqrt(n_power - 3))
print(f"\nMinimum |r| detectable at n={n_power}, alpha=0.05, power=0.80: {min_detectable_r:.3f}")

nearby_unconfirmed vs wqi_2024 (n=59): r=-0.023, p=0.864, 95% CI=[-0.277, 0.235]
nearby_unconfirmed vs wqi_change (n=59): r=-0.124, p=0.350, 95% CI=[-0.368, 0.137]

Minimum |r| detectable at n=59, alpha=0.05, power=0.80: 0.358


**Neither correlation is statistically distinguishable from zero** - both
95% confidence intervals comfortably span zero. But the more useful number
is the power analysis: at this sample size, only a true correlation of
|r| >= 0.36 or stronger could have been reliably detected. The observed
correlations (0.02, 0.12) are well below that threshold - meaning this
result genuinely can't distinguish "no relationship" from "a real but
modest relationship this sample is too small to see." That's a data
limitation, not evidence of no relationship, and it's the honest reason
this cross-reference doesn't get oversold as a finding.

## Step 6: Export for the map

In [12]:
EXPORT_COLS = ["STATION_NAME", "wqi_2023", "wqi_2024", "wqi_change",
               "nearby_outfalls", "nearby_unconfirmed", "mean_risk_score", "geometry"]
export = merged[EXPORT_COLS].round({
    "wqi_change": 1, "mean_risk_score": 1,
})
export.to_file("../docs/water_quality_change.geojson", driver="GeoJSON")

import json
limitations_summary = {
    "significance": sig_results,
    "min_detectable_r": round(float(min_detectable_r), 3),
    "n_power": int(n_power),
}
with open("../docs/limitations.json", "w") as f:
    json.dump(limitations_summary, f, indent=2)
print(json.dumps(limitations_summary, indent=2))

size_kb = os.path.getsize("../docs/water_quality_change.geojson") / 1024
print(f"water_quality_change.geojson: {size_kb:.0f} KB, {len(export)} stations")

{
  "significance": {
    "wqi_2024": {
      "r": -0.023,
      "p": 0.864,
      "n": 59,
      "ci_low": -0.277,
      "ci_high": 0.235
    },
    "wqi_change": {
      "r": -0.124,
      "p": 0.35,
      "n": 59,
      "ci_low": -0.368,
      "ci_high": 0.137
    }
  },
  "min_detectable_r": 0.358,
  "n_power": 59
}
water_quality_change.geojson: 16 KB, 59 stations
